# VJ146A2 Jan-Jun 2023 Selected-City Pixel Close-Ups

This notebook inspects five Jan-Jun 2023 large-settlement dark-event candidates from the VJ146A2 settlement-day panel.

The plots are deliberately **not settlement-aggregated**. They show raw 500 m VJ pixels in a close-up window, with the settlement polygon contour overlaid.

Selected Jan-Jun events only:

- Maluti-a-Phofung Local Municipality, strict-dark event date `2023-03-10`.
- Bushbuckridge, strict-dark event date `2023-05-25`.
- Polokwane Local Municipality, strict-dark event date `2023-05-19`.
- Nkomazi, mostly-dark event date `2023-04-17`.
- Thembisile Hani Local Municipality, mostly-dark event date `2023-01-23`.

Each selected settlement also has a satellite-context panel using Esri World Imagery tiles. The cyan contour is the settlement polygon used for aggregation.


## How To Read The Panels

Each settlement has two visual outputs:

1. **Satellite context panel**: Esri World Imagery basemap with the settlement contour in cyan.
2. **Raw VJ pixel close-up**: a 2 x 3 panel around the event date.

For the 2 x 3 panel:

- Columns: day before, event day, day after.
- Top row: valid radiance bins for the 500 m VJ pixels. This is the `rad` band after applying the same valid-pixel mask used by the binary panel.
- Bottom row: binary pixel class using the production threshold, after the same valid-pixel mask.
- Medium grey: invalid/missing pixels in both rows.
- Purple: valid but below or equal to the production threshold (`rad <= 1`).
- Yellow/orange/red in the top row: valid pixels above the threshold, split into radiance bins.
- Yellow in the bottom row: valid lit pixels (`rad > 1`).
- Cyan line: settlement contour.

This is meant to distinguish actual darkening from missing valid observations, geometry artifacts, or other QA issues.


In [ ]:
from pathlib import Path
import os
import subprocess

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        helper = candidate / "validation" / "closeups" / "scripts" / "vj146a2_jan_jun_selected_city_pixel_closeups.R"
        paths_helper = candidate / "validation" / "scripts" / "validation_paths.R"
        if helper.exists() and paths_helper.exists():
            return candidate
    raise FileNotFoundError("Could not locate Reliability-Assessment repo root with validation helpers")

REPO_ROOT = find_repo_root()
HELPER = REPO_ROOT / "validation" / "closeups" / "scripts" / "vj146a2_jan_jun_selected_city_pixel_closeups.R"
OUT_DIR = REPO_ROOT / "validation" / "closeups" / "figures" / "generated" / "vj146a2_jan_jun_selected_city_pixel_closeups"

paths = {
    "metrics": OUT_DIR / "vj146a2_jan_jun_selected_city_event_window_metrics.csv",
    "analysis": OUT_DIR / "vj146a2_jan_jun_selected_city_analysis_metrics.csv",
    "locations": OUT_DIR / "vj146a2_jan_jun_selected_city_locations.csv",
}

EVENTS = [
    {"settlement_id": "30250", "event_date": "2023-03-10", "label": "Maluti-a-Phofung Local Municipality"},
    {"settlement_id": "76354", "event_date": "2023-05-25", "label": "Bushbuckridge"},
    {"settlement_id": "73644", "event_date": "2023-05-19", "label": "Polokwane Local Municipality"},
    {"settlement_id": "69888", "event_date": "2023-04-17", "label": "Nkomazi"},
    {"settlement_id": "61860", "event_date": "2023-01-23", "label": "Thembisile Hani Local Municipality"},
]


def satellite_path(settlement_id, event_date):
    return OUT_DIR / f"satellite_closeup_{settlement_id}_{event_date}.png"


def pixel_path(settlement_id, event_date):
    return OUT_DIR / f"vj146a2_pixel_closeup_{settlement_id}_{event_date}.png"


## Regenerate Figures

Leave `RUN_R_HELPER = True` when you want to refresh the satellite panels and raw VJ close-ups. Set it to `False` if you only want to inspect already-generated outputs.

The helper fetches Esri World Imagery tiles the first time it runs and caches them under the output folder. If tile download fails, rerun with network access enabled or reuse an existing tile cache.


In [ ]:
RUN_R_HELPER = True

if RUN_R_HELPER:
    result = subprocess.run(
        ["Rscript", str(HELPER.relative_to(REPO_ROOT))],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)

missing = [str(p) for p in paths.values() if not p.exists()]
for event in EVENTS:
    missing.extend(
        str(p)
        for p in [satellite_path(event["settlement_id"], event["event_date"]), pixel_path(event["settlement_id"], event["event_date"])]
        if not p.exists()
    )
if missing:
    raise FileNotFoundError("Missing expected close-up outputs:\n" + "\n".join(missing))


## Event Metrics and Helpers

The event-window metric rows come from the Jan-Jun yearlykeep settlement-day panel. The images below are raw pixels; these rows are shown only to label the before/event/after dates.

The benchmark table under each image is computed over available Jan-Jun rows retained in the final VJ settlement-day panel.


In [ ]:
metrics = pd.read_csv(paths["metrics"], parse_dates=["date", "event_date", "local_overpass_date"])
analysis = pd.read_csv(paths["analysis"], parse_dates=["event_date", "first_panel_date", "last_panel_date"])
locations = pd.read_csv(paths["locations"], parse_dates=["event_date"])


def _fmt_value(value, digits=4):
    if pd.isna(value):
        return "NA"
    return f"{value:.{digits}f}"


def analysis_benchmark(settlement_id):
    row = analysis.loc[analysis["settlement_id"].astype(str).eq(str(settlement_id))]
    if row.empty:
        raise ValueError(f"No Jan-Jun metrics for settlement_id={settlement_id}")
    row = row.iloc[0]
    records = [
        ("available panel window", f"{row['first_panel_date'].date()} to {row['last_panel_date'].date()}"),
        ("days retained in panel", f"{int(row['n_days'])}"),
        ("days with p_lit", f"{int(row['n_days_with_p_lit'])}"),
        ("p_lit Jan-Jun mean", _fmt_value(row["analysis_p_lit_mean"])),
        ("p_lit Jan-Jun median", _fmt_value(row["analysis_p_lit_median"])),
        ("p_lit Jan-Jun min", _fmt_value(row["analysis_p_lit_min"])),
        ("coverage Jan-Jun mean", _fmt_value(row["analysis_coverage_mean"])),
        ("coverage Jan-Jun median", _fmt_value(row["analysis_coverage_median"])),
        ("daily mean rad Jan-Jun mean", _fmt_value(row["analysis_daily_mean_rad_mean"])),
        ("daily median rad Jan-Jun median", _fmt_value(row["analysis_daily_median_rad_median"])),
        ("strict-dark days", f"{int(row['analysis_strict_dark_count'])}"),
        ("mostly-dark days", f"{int(row['analysis_mostly_dark_count'])}"),
        ("selection reason", row["selection_reason"]),
    ]
    return pd.DataFrame(records, columns=["metric", "value"])


def settlement_location(settlement_id, event_date):
    row = locations.loc[
        locations["settlement_id"].astype(str).eq(str(settlement_id))
        & locations["event_date"].dt.strftime("%Y-%m-%d").eq(event_date)
    ]
    if row.empty:
        raise ValueError(f"No location metadata for settlement_id={settlement_id}, event_date={event_date}")
    cols = [
        "event_label",
        "event_class",
        "village_name",
        "admin_cgaz_1",
        "admin_cgaz_2",
        "population",
        "representative_lat",
        "representative_lon",
        "google_earth_search",
    ]
    return row[cols]


def event_window(settlement_id, event_date):
    rows = metrics.loc[
        metrics["settlement_id"].astype(str).eq(str(settlement_id))
        & metrics["event_date"].dt.strftime("%Y-%m-%d").eq(event_date)
    ].copy()
    cols = [
        "relative_day",
        "date",
        "local_overpass_date",
        "p_lit_sett",
        "coverage",
        "mean_rad_sett",
        "median_rad_sett",
        "mlr_mean_1_2am_primary",
        "shed_share_1_2am_primary",
    ]
    return rows[cols]


def show_event(settlement_id, event_date):
    display(Image(filename=str(satellite_path(settlement_id, event_date))))
    display(Image(filename=str(pixel_path(settlement_id, event_date))))
    display(settlement_location(settlement_id, event_date))
    display(event_window(settlement_id, event_date))
    display(analysis_benchmark(settlement_id))


## Maluti-a-Phofung Local Municipality, 2023-03-10

Strict-dark event. This is the largest Jan-Jun strict-dark candidate above 100k population, with acceptable coverage and high Eskom MLR exposure.


In [ ]:
show_event("30250", "2023-03-10")


## Bushbuckridge, 2023-05-25

Strict-dark event. Full coverage makes this a useful raw-pixel check, while the lower national shed share means it may reveal local outage, artifact, or non-MLR darkness.


In [ ]:
show_event("76354", "2023-05-25")


## Polokwane Local Municipality, 2023-05-19

Strict-dark event. Full coverage and a large drop from a high Jan-Jun median make this a strong pixel-confirmation candidate.


In [ ]:
show_event("73644", "2023-05-19")


## Nkomazi, 2023-04-17

Mostly-dark event. Very large settlement, full coverage, high baseline lit share, and high Eskom shed share.


In [ ]:
show_event("69888", "2023-04-17")


## Thembisile Hani Local Municipality, 2023-01-23

Mostly-dark event. Large settlement with full coverage and a sharp drop from a high Jan-Jun median.


In [ ]:
show_event("61860", "2023-01-23")


## Working Interpretation

Use these panels as a candidate-screening tool, not as confirmed outage evidence.

Initial hypotheses to check visually:

- **Maluti-a-Phofung**: strongest large-population strict-dark candidate; inspect whether the darkening is spatially coherent inside the contour.
- **Bushbuckridge**: full-coverage strict-dark event with relatively low national MLR; useful for identifying local outage, local non-grid effects, or a VJ/geometry artifact.
- **Polokwane**: full-coverage strict-dark event and large baseline drop; strong candidate if raw pixels darken coherently.
- **Nkomazi**: mostly-dark large settlement with full coverage and high MLR; useful bridge between October-style Nkomazi evidence and Jan-Jun behavior.
- **Thembisile Hani**: large mostly-dark candidate with full coverage; check whether the settlement is already partially dark before the event or shows a clear event-day transition.

After July-December yearlykeep panels are materialized, rerun the event scan over the full year and update this notebook's event list if stronger full-year candidates appear.
